In [1]:
import requests
from typing import Dict, List, Any

# -----------------------------------
# CONFIG
# -----------------------------------

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
WIKIPEDIA_SUMMARY_URL = "https://en.wikipedia.org/api/rest_v1/page/summary/"
OPEN_ELEVATION_URL = "https://api.open-elevation.com/api/v1/lookup"

HEADERS = {
    "User-Agent": "travel-destination-enricher/1.0"
}


# -----------------------------------
# ELEVATION
# -----------------------------------

def get_elevation(lat: float, lon: float) -> Dict[str, Any]:
    url = f"{OPEN_ELEVATION_URL}?locations={lat},{lon}"

    try:
        response = requests.get(url, headers=HEADERS, timeout=20)
        response.raise_for_status()

        data = response.json()

        elevation = data["results"][0]["elevation"]

        return {
            "elevation_m": elevation
        }

    except Exception as e:
        return {
            "elevation_m": None,
            "error": str(e)
        }


# -----------------------------------
# OVERPASS QUERY
# -----------------------------------

def build_overpass_query(lat: float, lon: float, radius_m: int) -> str:
    return f"""
    [out:json][timeout:25];

    (
      node(around:{radius_m},{lat},{lon})[tourism];
      way(around:{radius_m},{lat},{lon})[tourism];

      node(around:{radius_m},{lat},{lon})[natural];
      way(around:{radius_m},{lat},{lon})[natural];

      node(around:{radius_m},{lat},{lon})[historic];
      way(around:{radius_m},{lat},{lon})[historic];

      node(around:{radius_m},{lat},{lon})[leisure];
      way(around:{radius_m},{lat},{lon})[leisure];
    );

    out center tags;
    """


# -----------------------------------
# FETCH POIs
# -----------------------------------

def fetch_pois(lat: float, lon: float, zoom: int) -> List[Dict[str, Any]]:
    """
    Higher zoom -> smaller radius
    """

    zoom_radius_map = {
        5: 50000,
        8: 25000,
        10: 10000,
        12: 5000,
        14: 2500,
        16: 1000
    }

    radius = zoom_radius_map.get(zoom, 5000)

    query = build_overpass_query(lat, lon, radius)

    try:
        response = requests.post(
            OVERPASS_URL,
            data=query,
            headers=HEADERS,
            timeout=60
        )

        response.raise_for_status()

        data = response.json()

        pois = []

        for element in data.get("elements", []):

            tags = element.get("tags", {})

            poi = {
                "id": element.get("id"),
                "type": element.get("type"),
                "name": tags.get("name"),
                "category": (
                    tags.get("tourism")
                    or tags.get("natural")
                    or tags.get("historic")
                    or tags.get("leisure")
                ),
                "description": tags.get("description"),
                "wikidata": tags.get("wikidata"),
                "wikipedia": tags.get("wikipedia"),
                "website": tags.get("website"),
                "lat": element.get("lat") or element.get("center", {}).get("lat"),
                "lon": element.get("lon") or element.get("center", {}).get("lon"),
                "tags": tags
            }

            if poi["name"]:
                pois.append(poi)

        return pois

    except Exception as e:
        print("Overpass error:", e)
        return []


# -----------------------------------
# WIKIPEDIA SUMMARY
# -----------------------------------

def get_wikipedia_summary(title: str) -> Dict[str, Any]:
    """
    title example:
    'Manali'
    """

    try:
        url = WIKIPEDIA_SUMMARY_URL + title

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=20
        )

        response.raise_for_status()

        data = response.json()

        return {
            "title": data.get("title"),
            "summary": data.get("extract"),
            "thumbnail": (
                data.get("thumbnail", {}).get("source")
                if data.get("thumbnail")
                else None
            ),
            "url": (
                data.get("content_urls", {})
                .get("desktop", {})
                .get("page")
            )
        }

    except Exception as e:
        return {
            "error": str(e)
        }


# -----------------------------------
# DESTINATION ENRICHER
# -----------------------------------

def enrich_destination(
    destination: str,
    lat: float,
    lon: float,
    zoom: int
) -> Dict[str, Any]:

    elevation_data = get_elevation(lat, lon)

    pois = fetch_pois(lat, lon, zoom)

    wiki_summary = get_wikipedia_summary(destination)

    famous_places = []

    for poi in pois[:10]:
        famous_places.append({
            "name": poi["name"],
            "category": poi["category"],
            "wikidata": poi["wikidata"]
        })

    return {
        "destination": destination,
        "coordinates": {
            "lat": lat,
            "lon": lon
        },
        "zoom": zoom,
        "elevation": elevation_data,
        "summary": wiki_summary,
        "pois_count": len(pois),
        "famous_places": famous_places,
        "pois": pois
    }


# -----------------------------------
# EXAMPLE
# -----------------------------------

if __name__ == "__main__":

    result = enrich_destination(
        destination="Manali",
        lat=32.2432,
        lon=77.1892,
        zoom=12
    )

    import json

    print(json.dumps(result, indent=2, ensure_ascii=False))

Overpass error: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
{
  "destination": "Manali",
  "coordinates": {
    "lat": 32.2432,
    "lon": 77.1892
  },
  "zoom": 12,
  "elevation": {
    "elevation_m": null,
    "error": "504 Server Error: Gateway Time-out for url: https://api.open-elevation.com/api/v1/lookup?locations=32.2432,77.1892"
  },
  "summary": {
    "title": "Manali",
    "summary": "Manali may refer to:",
    "thumbnail": null,
    "url": "https://en.wikipedia.org/wiki/Manali"
  },
  "pois_count": 0,
  "famous_places": [],
  "pois": []
}


In [2]:
import requests
import random
import time
from typing import Dict, List, Any


# ---------------------------------------------------
# CONFIG
# ---------------------------------------------------

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://z.overpass-api.de/api/interpreter",
]

OPENTOPO_URL = "https://api.opentopodata.org/v1/aster30m"

WIKI_SEARCH_URL = "https://en.wikipedia.org/w/api.php"
WIKI_SUMMARY_URL = "https://en.wikipedia.org/api/rest_v1/page/summary/"

HEADERS = {
    "User-Agent": "travel-enricher/1.0"
}


# ---------------------------------------------------
# RETRY HELPER
# ---------------------------------------------------

def retry_request(
    method,
    url,
    retries=3,
    delay=2,
    **kwargs
):

    for attempt in range(retries):

        try:
            response = requests.request(
                method,
                url,
                timeout=60,
                **kwargs
            )

            response.raise_for_status()

            return response

        except Exception as e:

            print(f"Attempt {attempt+1} failed: {e}")

            if attempt < retries - 1:
                time.sleep(delay)

    raise Exception(f"Failed after {retries} retries")


# ---------------------------------------------------
# ELEVATION
# ---------------------------------------------------

def get_elevation(lat: float, lon: float):

    url = f"{OPENTOPO_URL}?locations={lat},{lon}"

    try:

        response = retry_request(
            "GET",
            url,
            headers=HEADERS
        )

        data = response.json()

        return {
            "elevation_m": data["results"][0]["elevation"],
            "dataset": "aster30m"
        }

    except Exception as e:

        return {
            "elevation_m": None,
            "error": str(e)
        }


# ---------------------------------------------------
# OVERPASS
# ---------------------------------------------------

def build_overpass_query(
    lat: float,
    lon: float,
    radius_m: int
):

    return f"""
    [out:json][timeout:20];

    (
      node(around:{radius_m},{lat},{lon})[tourism=attraction];
      node(around:{radius_m},{lat},{lon})[natural];
      node(around:{radius_m},{lat},{lon})[historic];

      way(around:{radius_m},{lat},{lon})[tourism=attraction];
      way(around:{radius_m},{lat},{lon})[natural];
      way(around:{radius_m},{lat},{lon})[historic];
    );

    out center tags 50;
    """


def fetch_pois(
    lat: float,
    lon: float,
    zoom: int
):

    zoom_radius_map = {
        8: 20000,
        10: 10000,
        12: 5000,
        14: 2500,
        16: 1000
    }

    radius = zoom_radius_map.get(zoom, 5000)

    query = build_overpass_query(
        lat,
        lon,
        radius
    )

    random.shuffle(OVERPASS_ENDPOINTS)

    for endpoint in OVERPASS_ENDPOINTS:

        try:

            response = retry_request(
                "POST",
                endpoint,
                data=query,
                headers=HEADERS
            )

            data = response.json()

            pois = []

            for element in data.get("elements", []):

                tags = element.get("tags", {})

                name = tags.get("name")

                if not name:
                    continue

                poi = {
                    "name": name,
                    "category": (
                        tags.get("tourism")
                        or tags.get("natural")
                        or tags.get("historic")
                    ),
                    "description": tags.get("description"),
                    "wikidata": tags.get("wikidata"),
                    "wikipedia": tags.get("wikipedia"),
                    "lat": (
                        element.get("lat")
                        or element.get("center", {}).get("lat")
                    ),
                    "lon": (
                        element.get("lon")
                        or element.get("center", {}).get("lon")
                    )
                }

                pois.append(poi)

            return pois

        except Exception as e:

            print(f"Overpass endpoint failed: {endpoint}")
            print(e)

    return []


# ---------------------------------------------------
# WIKIPEDIA
# ---------------------------------------------------

def resolve_wikipedia_title(destination: str):

    params = {
        "action": "opensearch",
        "search": destination,
        "limit": 1,
        "namespace": 0,
        "format": "json"
    }

    response = retry_request(
        "GET",
        WIKI_SEARCH_URL,
        params=params,
        headers=HEADERS
    )

    data = response.json()

    titles = data[1]

    if not titles:
        return None

    return titles[0]


def get_wikipedia_summary(destination: str):

    try:

        title = resolve_wikipedia_title(destination)

        if not title:
            return None

        url = WIKI_SUMMARY_URL + title

        response = retry_request(
            "GET",
            url,
            headers=HEADERS
        )

        data = response.json()

        return {
            "title": data.get("title"),
            "summary": data.get("extract"),
            "thumbnail": (
                data.get("thumbnail", {})
                .get("source")
            ),
            "url": (
                data.get("content_urls", {})
                .get("desktop", {})
                .get("page")
            )
        }

    except Exception as e:

        return {
            "error": str(e)
        }


# ---------------------------------------------------
# MAIN
# ---------------------------------------------------

def enrich_destination(
    destination,
    lat,
    lon,
    zoom
):

    elevation = get_elevation(lat, lon)

    pois = fetch_pois(lat, lon, zoom)

    wiki = get_wikipedia_summary(
        f"{destination}, Himachal Pradesh"
    )

    famous_places = [
        {
            "name": p["name"],
            "category": p["category"]
        }
        for p in pois[:10]
    ]

    return {
        "destination": destination,
        "coordinates": {
            "lat": lat,
            "lon": lon
        },
        "elevation": elevation,
        "summary": wiki,
        "pois_count": len(pois),
        "famous_places": famous_places,
        "pois": pois
    }


# ---------------------------------------------------
# RUN
# ---------------------------------------------------

if __name__ == "__main__":

    result = enrich_destination(
        destination="Manali",
        lat=32.2432,
        lon=77.1892,
        zoom=12
    )

    import json

    print(json.dumps(
        result,
        indent=2,
        ensure_ascii=False
    ))

{
  "destination": "Manali",
  "coordinates": {
    "lat": 32.2432,
    "lon": 77.1892
  },
  "elevation": {
    "elevation_m": 1910.0,
    "dataset": "aster30m"
  },
  "summary": {
    "title": "Manali, Himachal Pradesh",
    "summary": "Manali is a resort town, near Kullu town in the Kullu district in the Indian state of Himachal Pradesh. It is situated at the northern end of the Kullu Valley, formed by the Beas River. The town is located in the Kullu district, approximately 270 kilometres (170 mi) north of the state capital of Shimla and 544 kilometres (338 mi) northeast of the national capital of New Delhi. Manali is a popular tourist destination in India and serves as the gateway to the Lahaul and Spiti district as well as the city of Leh in Ladakh.",
    "thumbnail": "https://upload.wikimedia.org/wikipedia/commons/thumb/0/03/Manali_City.jpg/330px-Manali_City.jpg",
    "url": "https://en.wikipedia.org/wiki/Manali%2C_Himachal_Pradesh"
  },
  "pois_count": 17,
  "famous_places": [
 

In [1]:
import requests
import random
import time


OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://z.overpass-api.de/api/interpreter",
]

HEADERS = {
    "User-Agent": "smart-poi-engine/1.0"
}


CATEGORY_MAP = {

    "attraction": [
        '["tourism"="attraction"]',
        '["tourism"="viewpoint"]',
        '["tourism"="museum"]'
    ],

    "park": [
        '["leisure"="park"]',
        '["boundary"="national_park"]'
    ],

    "cafe": [
        '["amenity"="cafe"]'
    ],

    "restaurant": [
        '["amenity"="restaurant"]'
    ],

    "hotel": [
        '["tourism"="hotel"]',
        '["tourism"="guest_house"]'
    ],

    "waterfall": [
        '["natural"="waterfall"]'
    ]
}


# ---------------------------------------------------
# RETRY
# ---------------------------------------------------

def retry_request(method, url, retries=3, delay=2, **kwargs):

    for attempt in range(retries):

        try:

            response = requests.request(
                method,
                url,
                timeout=60,
                **kwargs
            )

            response.raise_for_status()

            return response

        except Exception as e:

            print(f"Attempt {attempt+1} failed:", e)

            if attempt < retries - 1:
                time.sleep(delay)

    raise Exception("Request failed")


# ---------------------------------------------------
# QUERY BUILDER
# ---------------------------------------------------

def build_query(lat, lon, radius_m, category):

    filters = CATEGORY_MAP.get(category.lower())

    if not filters:
        raise ValueError(f"Unsupported category: {category}")

    blocks = []

    for f in filters:

        blocks.append(f"""
            node(around:{radius_m},{lat},{lon}){f};
            way(around:{radius_m},{lat},{lon}){f};
            relation(around:{radius_m},{lat},{lon}){f};
        """)

    body = "\n".join(blocks)

    return f"""
    [out:json][timeout:25];

    (
        {body}
    );

    out center tags 200;
    """


# ---------------------------------------------------
# SMART SCORING
# ---------------------------------------------------

def calculate_poi_score(tags):

    score = 0

    # ----------------------------
    # Strong importance signals
    # ----------------------------

    if tags.get("wikidata"):
        score += 40

    if tags.get("wikipedia"):
        score += 40

    if tags.get("tourism"):
        score += 25

    if tags.get("heritage"):
        score += 30

    # ----------------------------
    # Rich metadata
    # ----------------------------

    if tags.get("description"):
        score += 20

    if tags.get("website"):
        score += 15

    if tags.get("image"):
        score += 20

    if tags.get("opening_hours"):
        score += 5

    if tags.get("phone"):
        score += 5

    # ----------------------------
    # Rich tag count
    # ----------------------------

    score += min(len(tags), 20)

    # ----------------------------
    # Named entities matter
    # ----------------------------

    name = tags.get("name", "")

    if len(name.split()) >= 2:
        score += 5

    # ----------------------------
    # Relation objects
    # usually larger/important
    # ----------------------------

    return score


# ---------------------------------------------------
# SEARCH
# ---------------------------------------------------

def search_places(
    lat,
    lon,
    category,
    radius_m=5000,
    limit=20
):

    query = build_query(
        lat,
        lon,
        radius_m,
        category
    )

    random.shuffle(OVERPASS_ENDPOINTS)

    for endpoint in OVERPASS_ENDPOINTS:

        try:

            response = retry_request(
                "POST",
                endpoint,
                data=query,
                headers=HEADERS
            )

            data = response.json()

            results = []

            for element in data.get("elements", []):

                tags = element.get("tags", {})

                name = tags.get("name")

                if not name:
                    continue

                score = calculate_poi_score(tags)

                poi = {

                    "name": name,

                    "score": score,

                    "lat": (
                        element.get("lat")
                        or element.get("center", {}).get("lat")
                    ),

                    "lon": (
                        element.get("lon")
                        or element.get("center", {}).get("lon")
                    ),

                    "category": category,

                    "description": tags.get("description"),

                    "website": tags.get("website"),

                    "wikidata": tags.get("wikidata"),

                    "wikipedia": tags.get("wikipedia"),

                    "opening_hours": tags.get("opening_hours"),

                    "tags": tags
                }

                results.append(poi)

            # --------------------------------
            # DEDUPE
            # --------------------------------

            deduped = {}

            for r in results:

                existing = deduped.get(r["name"])

                if (
                    not existing
                    or r["score"] > existing["score"]
                ):
                    deduped[r["name"]] = r

            # --------------------------------
            # SORT BY QUALITY
            # --------------------------------

            ranked = sorted(
                deduped.values(),
                key=lambda x: x["score"],
                reverse=True
            )

            return ranked[:limit]

        except Exception as e:

            print("Endpoint failed:", endpoint)
            print(e)

    return []


# ---------------------------------------------------
# EXAMPLE
# ---------------------------------------------------

if __name__ == "__main__":

    results = search_places(
        lat=48.8534,
        lon=48.8566,
        category="attraction",
        radius_m=15000,
        limit=20
    )

    import json

    print(json.dumps(
        results,
        indent=2,
        ensure_ascii=False
    ))

[]


In [3]:
import requests
import math


# ---------------------------------------------------
# CONFIG
# ---------------------------------------------------

HEADERS = {
    "User-Agent": "travel-discovery-engine/1.0"
}

WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_SUMMARY_API = "https://en.wikipedia.org/api/rest_v1/page/summary/"


# ---------------------------------------------------
# QUERY → WIKIPEDIA SEARCH PROFILE
# ---------------------------------------------------

QUERY_MAP = {

    "tourist attraction": {
        "radius": 10000,
        "limit": 20
    },

    "park": {
        "radius": 8000,
        "limit": 20
    },

    "cafe": {
        "radius": 5000,
        "limit": 20
    },

    "restaurant": {
        "radius": 5000,
        "limit": 20
    },

    "museum": {
        "radius": 10000,
        "limit": 20
    },

    "hotel": {
        "radius": 7000,
        "limit": 20
    }
}


# ---------------------------------------------------
# DISTANCE
# ---------------------------------------------------

def haversine(lat1, lon1, lat2, lon2):

    R = 6371

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c


# ---------------------------------------------------
# WIKIPEDIA GEOSEARCH
# ---------------------------------------------------

def geosearch_wikipedia(
    lat,
    lon,
    query="tourist attraction"
):

    profile = QUERY_MAP.get(
        query.lower(),
        QUERY_MAP["tourist attraction"]
    )

    params = {
        "action": "query",
        "list": "geosearch",
        "gscoord": f"{lat}|{lon}",
        "gsradius": profile["radius"],
        "gslimit": profile["limit"],
        "format": "json"
    }

    response = requests.get(
        WIKI_API,
        params=params,
        headers=HEADERS,
        timeout=20
    )

    response.raise_for_status()

    data = response.json()

    return data["query"]["geosearch"]


# ---------------------------------------------------
# PAGE SUMMARY
# ---------------------------------------------------

def fetch_summary(title):

    url = WIKI_SUMMARY_API + title

    try:

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=20
        )

        response.raise_for_status()

        data = response.json()

        return {

            "title": data.get("title"),

            "summary": data.get("extract"),

            "image": (
                data.get("thumbnail", {})
                .get("source")
            ),

            "url": (
                data.get("content_urls", {})
                .get("desktop", {})
                .get("page")
            )
        }

    except Exception:

        return None


# ---------------------------------------------------
# SMART FILTER
# ---------------------------------------------------

def is_relevant_place(place, query):

    title = place["title"].lower()

    q = query.lower()

    relevance_map = {

        "cafe": [
            "cafe",
            "coffee"
        ],

        "restaurant": [
            "restaurant",
            "food"
        ],

        "park": [
            "park",
            "garden"
        ],

        "museum": [
            "museum"
        ],

        "hotel": [
            "hotel",
            "resort"
        ]
    }

    keywords = relevance_map.get(q)

    if not keywords:
        return True

    for k in keywords:
        if k in title:
            return True

    return False


# ---------------------------------------------------
# MAIN SEARCH
# ---------------------------------------------------

def search_places(
    lat,
    lon,
    query="tourist attraction"
):

    raw_places = geosearch_wikipedia(
        lat,
        lon,
        query
    )

    results = []

    for place in raw_places:

        summary = fetch_summary(place["title"])

        if not summary:
            continue

        enriched = {

            "name": summary["title"],

            "summary": summary["summary"],

            "image": summary["image"],

            "url": summary["url"],

            "lat": place["lat"],

            "lon": place["lon"],

            "distance_km": round(
                haversine(
                    lat,
                    lon,
                    place["lat"],
                    place["lon"]
                ),
                2
            )
        }

        results.append(enriched)

    # ------------------------------------------------
    # OPTIONAL CATEGORY FILTER
    # ------------------------------------------------

    if query.lower() != "tourist attraction":

        filtered = []

        for r in results:

            if is_relevant_place(r, query):
                filtered.append(r)

        if filtered:
            results = filtered

    # ------------------------------------------------
    # SORT BY DISTANCE
    # ------------------------------------------------

    results = sorted(
        results,
        key=lambda x: x["distance_km"]
    )

    return results


# ---------------------------------------------------
# EXAMPLE
# ---------------------------------------------------

if __name__ == "__main__":

    results = search_places(
        lat=32.2432,
        lon=77.1892,
        query="cafe"
    )

    import json

    print(json.dumps(
        results[:10],
        indent=2,
        ensure_ascii=False
    ))

KeyError: 'title'